# Review cloud reference labels locally

Use this notebook in VS Code with the Jupyter extension. It reads the original Maxar TIFFs from this repository's local `data/` folder only when creating the review set. After the coordinator commits and pushes `data/reference-review-v2/`, reviewers can pull it and review the saved 1024 x 1024 patches without having the original TIFFs.

Review edits are saved under `data/reference-review-v2/annotations/<reviewer_id>/`. Use a distinct stable reviewer ID for each person. Commit and push that person's annotation folder to share progress.

## 1. Install review dependencies

Run this once in the selected VS Code Python environment. Restart the notebook kernel if VS Code requests it.

In [ ]:
%pip install rasterio matplotlib ipywidgets ipympl pillow numpy

## 2. Create the shared patch set — coordinator only

Run this once from the coordinator's checkout, which needs the source GeoTIFFs under `data/`. It creates a small patch bundle and candidate masks inside the repository. Do not rerun it after the folder exists; share that folder through Git instead. The raw TIFFs remain ignored by Git.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "data").is_dir():
    raise FileNotFoundError("Open this notebook from the repository workspace in VS Code.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from notebooks.review_reference_labels import prepare_review_set

DATA_DIR = REPO_ROOT / "data"
REVIEW_DIR = DATA_DIR / "reference-review-v2"
review_dir = prepare_review_set(
    input_dir=DATA_DIR,
    output_dir=REVIEW_DIR,
    patch_size=1024,
    patches_per_tile=3,
    minimum_valid_fraction=0.80,
    cloud_free_events=("Brazil-Flooding-May24",),
    seed=42,
)
print(f"Created review set: {review_dir}")

## 3. Review patches

The coordinator should commit and push the generated `data/reference-review-v2/` folder first. Other reviewers should pull those changes, skip the generation cell above, and run this cell. Each person's annotations are saved separately, so reviewers can review the same patches independently. Save progress by committing and pushing your own `annotations/<reviewer_id>/` folder.

In [ ]:
%matplotlib widget

from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "data").is_dir():
    raise FileNotFoundError("Open this notebook from the repository workspace in VS Code.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from notebooks.review_reference_labels import launch_reviewer

REVIEW_DIR = REPO_ROOT / "data" / "reference-review-v2"
# Change this to your own stable ID; use letters, numbers, underscores, or hyphens.
REVIEWER_ID = "reviewer1"
launch_reviewer(REVIEW_DIR, reviewer_id=REVIEWER_ID)

## Labeling reminders

- `0` = clear/non-cloud; `1` = cloud; `255` = ignore.
- Smoke and bright ground are non-cloud; use their tags for error analysis.
- Mark genuinely ambiguous pixels as ignore. Label identifiable thin cloud as cloud and tag it.
- A patch only counts as human-reviewed after you enter your name and click **Mark reviewed**.
- Reviewers' files are independent. Compare overlapping reviews and resolve disagreements before using labels as the final reference set.
- Do not commit or force-add raw `.tif` or `.gpkg` files; the repository ignores them.